In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
import pickle

df = pd.read_csv('../data/churn_cleaned.csv')
print("Shape:", df.shape)
df.head()

Shape: (10000, 11)


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [2]:
le = LabelEncoder()

# Binary encode Gender
df['Gender'] = le.fit_transform(df['Gender'])  # Male=1, Female=0

# One-hot encode Geography
df = pd.get_dummies(df, columns=['Geography'], drop_first=False)

print("Columns after encoding:\n", df.columns.tolist())
df.head()

Columns after encoding:
 ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,True,False,False
1,608,0,41,1,83807.86,1,0,1,112542.58,0,False,False,True
2,502,0,42,8,159660.80,3,1,0,113931.57,1,True,False,False
3,699,0,39,1,0.00,2,0,0,93826.63,0,True,False,False
4,850,0,43,2,125510.82,1,1,1,79084.10,0,False,False,True


In [3]:
X = df.drop(columns=['Exited'])
y = df['Exited']

print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts())

Features shape: (10000, 12)
Target distribution:
 Exited
0    7963
1    2037
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # Keeps churn ratio balanced in both splits
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain churn rate:", round(y_train.mean() * 100, 2), "%")
print("Test churn rate:", round(y_test.mean() * 100, 2), "%")

Train size: (8000, 12)
Test size: (2000, 12)

Train churn rate: 20.38 %
Test churn rate: 20.35 %


In [5]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", y_train_sm.value_counts().to_dict())

Before SMOTE: {0: 6370, 1: 1630}
After SMOTE:  {1: 6370, 0: 6370}


In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)          # Only transform, never fit on test

print("✅ Scaling done!")
print("Train scaled shape:", X_train_scaled.shape)
print("Test scaled shape: ", X_test_scaled.shape)

✅ Scaling done!
Train scaled shape: (12740, 12)
Test scaled shape:  (2000, 12)


In [7]:
# Save processed data
np.save('../data/X_train.npy', X_train_scaled)
np.save('../data/X_test.npy',  X_test_scaled)
np.save('../data/y_train.npy', y_train_sm.values)
np.save('../data/y_test.npy',  y_test.values)

# Save scaler & feature names for later export
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names for Power BI export later
feature_names = X.columns.tolist()
pd.Series(feature_names).to_csv('../data/feature_names.csv', index=False)

print("✅ All preprocessing artifacts saved!")
print("\nFeatures used:", feature_names)

✅ All preprocessing artifacts saved!

Features used: ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']
